# CS-781 Phase 3 (Rhetorical Roles) - ENSEMBLE, bias calibration and error analysis (`p3s_ensemble`)

**No GPU and no Phase-1 model are needed.** This notebook reads files that the sweep notebooks already saved in the bucket:
* the **out-of-fold predictions** of every cross-validated setup (each document's prediction from a model that never saw it), and
* the **test log-probabilities** of every refit model.

It then (1) scores each member, simple averages and a greedy selection on the out-of-fold predictions, (2) learns per-label offsets that help rare labels and estimates their real gain honestly (learned on 4 folds, applied to the 5th), (3) writes the submission zips `ens` and `ens_bias` from the test log-probabilities, (4) prints the error analysis for the report: CL vs IT per label, closely related roles (petitioner vs respondent arguments, the three precedent roles, rulings vs reasoning) and Dissent, (5) compares every member (every technique) on the related roles, and (6) tabulates size and compute (parameters, stored size, minutes per fold, peak memory, tokens per second) for the main setups.
Edit `MEMBERS` in cell 1 to add or remove members; members whose files are missing are skipped. **Kaggle:** Internet on, secrets `HF` and `HF_WRITE` attached (a CPU session is enough).

## 1. Settings

In [1]:
# [P3-PRELUDE:settings]
import json, numpy as np
SWEEP_NAME = "p3s_ensemble"
QUICK_TEST = False
# Members of the ensemble: (label, cache prefix of the cross-validation files, setup name, file with the test log-probabilities, setup name in that file).
# The cross-validation files hold every setup's held-out (out-of-fold) predictions; the test file holds the log-probabilities of the
# refit model on the 10 test documents. Members whose files are missing are skipped, so this cell can be run before every sweep has finished.
MEMBERS = [
    ("moe_full_ft",        "p3s_moe", "full_ft",           "test_logprobs_p3s_moe.pt",     "full_ft"),
    ("moe_lora",           "p3s_moe", "lora",              "test_logprobs_p3s_moe.pt",     "lora"),
    ("moe_lorma",          "p3s_moe", "lorma",             "test_logprobs_p3s_moe.pt",     "lorma"),
    ("moe_ia3",            "p3s_moe", "ia3",               "test_logprobs_p3s_moe_ctx.pt", "ia3"),
    ("moe_lora_seq48",     "p3s_moe", "ctx_seq48",         "test_logprobs_p3s_moe_ctx.pt", "ctx_seq48"),
    ("moe_lora_win2048",   "p3s_moe", "ctx_win2048_seq64", "test_logprobs_p3s_moe_ctx.pt", "ctx_win2048_seq64"),
    ("moe_lorma_seq48",    "p3s_moe", "lorma_s48",         "test_logprobs_p3s_moe_s48.pt", "lorma_s48"),
    ("moe_full_ft_seq48",  "p3s_moe", "full_ft_s48",       "test_logprobs_p3s_moe_s48.pt", "full_ft_s48"),
    ("moe_rslora_seq48",   "p3s_moe", "rslora_s48",        "test_logprobs_p3s_moe_s48.pt", "rslora_s48"),
    ("moe_ia3_seq48",      "p3s_moe", "ia3_s48",           "test_logprobs_p3s_moe_s48.pt", "ia3_s48"),
    ("moe_pos_lora",       "p3s_moe", "pos_lora_s48",      "test_logprobs_p3s_moe_pos.pt", "pos_lora_s48"),
    ("moe_pos_lorma",      "p3s_moe", "pos_lorma_s48",     "test_logprobs_p3s_moe_pos.pt", "pos_lorma_s48"),
    ("moe_frozen_s48",     "p3s_moe", "frozen_s48",        "test_logprobs_p3s_moe_frozen.pt", "frozen_s48"),
    ("moe_cw_inv",         "p3s_moe", "abl_cw_inv",        "test_logprobs_p3s_moe_ablate.pt", "abl_cw_inv"),
    ("gqa_full_ft_seq48",  "p3s_gqa160", "full_ft_s48",    "test_logprobs_p3s_gqa160_s48.pt", "full_ft_s48"),
    ("gqa_qlora_seq48",    "p3s_gqa160", "qlora_s48",      "test_logprobs_p3s_gqa160_s48.pt", "qlora_s48"),
    ("moe_pos_dom_lora", "p3s_moe", "pos_dom_lora", "test_logprobs_p3s_moe_combo1.pt", "pos_dom_lora"),
    ("moe_pos_cwinv_lora", "p3s_moe", "pos_cwinv_lora", "test_logprobs_p3s_moe_combo1.pt", "pos_cwinv_lora"),
    ("moe_pos_lora_avg", "p3s_moe", "pos_lora_avg", "test_logprobs_p3s_moe_combo1.pt", "pos_lora_avg"),
    ("moe_pos_lora_ls", "p3s_moe", "pos_lora_ls", "test_logprobs_p3s_moe_combo1.pt", "pos_lora_ls"),
    ("moe_pos_lora_seed1", "p3s_moe", "pos_lora_seed1", "test_logprobs_p3s_moe_combo1.pt", "pos_lora_seed1"),
    ("moe_pos_lora_seed2", "p3s_moe", "pos_lora_seed2", "test_logprobs_p3s_moe_combo1.pt", "pos_lora_seed2"),
    ("moe_pos_ia3", "p3s_moe", "pos_ia3", "test_logprobs_p3s_moe_combo2.pt", "pos_ia3"),
    ("moe_pos_frozen", "p3s_moe", "pos_frozen", "test_logprobs_p3s_moe_combo2.pt", "pos_frozen"),
    ("moe_pos_dom_ia3", "p3s_moe", "pos_dom_ia3", "test_logprobs_p3s_moe_combo2.pt", "pos_dom_ia3"),
    ("moe_pos_rslora", "p3s_moe", "pos_rslora", "test_logprobs_p3s_moe_combo2.pt", "pos_rslora"),
    ("moe_pos_qlora", "p3s_moe", "pos_qlora", "test_logprobs_p3s_moe_combo2.pt", "pos_qlora"),
    ("moe_pos_adalora", "p3s_moe", "pos_adalora", "test_logprobs_p3s_moe_combo2.pt", "pos_adalora"),
    ("moe_pos_lora_s96", "p3s_moe", "pos_lora_s96", "test_logprobs_p3s_moe_long.pt", "pos_lora_s96"),
    ("moe_pos_lora_s128", "p3s_moe", "pos_lora_s128", "test_logprobs_p3s_moe_long.pt", "pos_lora_s128"),
    ("moe_pos_ia3_s96", "p3s_moe", "pos_ia3_s96", "test_logprobs_p3s_moe_long.pt", "pos_ia3_s96"),
]
K_FOLDS = 5
BIAS_GRID = [round(x, 2) for x in np.arange(-1.5, 1.51, 0.25)]   # additive per-label offsets tried on the log-probabilities
BIAS_ROUNDS = 3
MIN_GAIN = 0.002        # greedy member selection adds a member only if the out-of-fold macro-F1 improves by more than this
QUICK = QUICK_TEST

# Setups whose size / speed / memory are tabulated (read from the cached cross-validation files; pooled groups only): (label, cache prefix, setup)
EFF_SETUPS = [
    ("MoE full_ft", "p3s_moe", "full_ft"), ("MoE lora", "p3s_moe", "lora"), ("MoE lorma", "p3s_moe", "lorma"), ("MoE ia3", "p3s_moe", "ia3"),
    ("MoE lora 48 sent", "p3s_moe", "ctx_seq48"), ("MoE lora win2048", "p3s_moe", "ctx_win2048_seq64"),
    ("MoE lora 48 + position", "p3s_moe", "pos_lora_s48"), ("MoE frozen 48", "p3s_moe", "frozen_s48"), ("MoE ia3 48", "p3s_moe", "ia3_s48"),
    ("MoE random-init frozen 48", "p3s_moe", "rand_frozen_s48"), ("MoE random-init full_ft 48", "p3s_moe", "rand_full_ft_s48"), ("MoE KD lora 48 + position", "p3s_moe", "kd_pos_lora"),
    ("GQA full_ft 48", "p3s_gqa160", "full_ft_s48"), ("GQA qlora 48", "p3s_gqa160", "qlora_s48"), ("LSTM full_ft 48", "p3s_lstm", "full_ft_s48"),
    ("GQA lora", "p3s_gqa160", "lora_pooled"), ("GQA rslora", "p3s_gqa160", "rslora"), ("GQA lorma", "p3s_gqa160", "lorma"), ("GQA qlora", "p3s_gqa160", "qlora"),
    ("GQA adalora", "p3s_gqa160", "adalora"), ("GQA ia3", "p3s_gqa160", "ia3"), ("GQA prompt", "p3s_gqa160", "prompt"), ("GQA prefix", "p3s_gqa160", "prefix"),
    ("GQA full_ft", "p3s_gqa160", "full_ft"), ("GQA frozen head", "p3s_gqa160", "frozen_head_only"),
    ("GQA KD lora", "p3s_kd_gqa160", "kd_lora"), ("GQA KD full_ft", "p3s_kd_gqa160", "kd_full_ft"),
    ("LSTM full_ft", "p3s_lstm", "full_ft"), ("LSTM frozen head", "p3s_lstm", "frozen_head_only"), ("LSTM KD full_ft", "p3s_kd_lstm", "kd_full_ft"),
    ("MoR lorma", "p3s_mor", "lorma"), ("MoR lora", "p3s_mor", "lora"), ("MoR full_ft", "p3s_mor", "full_ft"),
    ("GDN lorma", "p3s_gdn", "lorma"), ("GDN lora", "p3s_gdn", "lora"), ("GDN ia3", "p3s_gdn", "ia3"), ("GDN full_ft", "p3s_gdn", "full_ft"),
]


## 2. Set-up: login, bucket

In [2]:
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cuda Tesla T4


## 3. Phase-3 engine (scoring, submission writer)

In [3]:
# ===================== Phase 3 (RR) engine: sentence units, pooling, heads, folds, training, inference, scoring =====================
# Works with any Phase-1 model wrapped as a "backbone" exposing  features(ids, mask, soft_prompt) -> (B, T, d_model).
# Data model: a document = ordered sentences; a "unit" = up to SEQ_SENTS consecutive sentences of one document. The backbone reads the
# sentences of a unit either one by one (PACK="alone") or packed together into windows of <= WINDOW tokens (PACK="together");
# every sentence is pooled into one vector; an optional BiLSTM runs over the sentence vectors of the unit; a linear layer gives 13 scores.
import math, time, random, json, os, copy, zipfile
from collections import Counter, defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F

IGNORE = -100


# ----------------------------------------------------------------------------------------------------------------------
# 1. Labels and documents
# ----------------------------------------------------------------------------------------------------------------------
def labels_from_vocab(vocab_raw):
    """label_vocab.json is a dict label -> int (Fact 0 ... None 12). Returns the label list ordered by that int."""
    if isinstance(vocab_raw, dict):
        vals = list(vocab_raw.values())
        if all(isinstance(v, int) for v in vals):
            labels = [k for k, _ in sorted(vocab_raw.items(), key=lambda kv: kv[1])]
        else:
            labels = list(vocab_raw.keys())
    else:
        labels = list(vocab_raw)
    assert all(isinstance(l, str) for l in labels), "a label is not a string (the 'None' problem?)"
    assert "None" in labels and len(set(labels)) == len(labels)
    return labels


def as_doc_list(obj, domain):
    """Train/dev files: list of {id, text, labels}. Test files: dict {id: [sentences]}. Returns [{id, domain, text, labels|None}]."""
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            out.append(dict(id=k, domain=domain, text=list(v), labels=None))
    else:
        for r in obj:
            out.append(dict(id=r["id"], domain=domain, text=list(r["text"]), labels=list(r["labels"]) if r.get("labels") is not None else None))
    return out


def encode_docs(tok, docs, label2id, max_sent_tokens):
    """Tokenise every sentence with the Phase-1 BPE (sentences longer than max_sent_tokens keep their first tokens)."""
    out, n_trunc, n_sent = [], 0, 0
    for d in docs:
        sents = []
        for s in d["text"]:
            ids = tok.encode(s)
            if not ids:
                ids = [1]                                        # <unk>: a sentence must have at least one token
            if len(ids) > max_sent_tokens:
                ids = ids[:max_sent_tokens]; n_trunc += 1
            sents.append(ids); n_sent += 1
        y = None if d.get("labels") is None else [label2id[l] for l in d["labels"]]
        if y is not None:
            assert len(y) == len(sents), f"{d['id']}: {len(y)} labels for {len(sents)} sentences"
        out.append(dict(id=d["id"], domain=d["domain"], sents=sents, y=y, n=len(sents)))
    return out, dict(sentences=n_sent, truncated=n_trunc)


# ----------------------------------------------------------------------------------------------------------------------
# 2. Units, packing, collation
# ----------------------------------------------------------------------------------------------------------------------
def make_units(n, seq_sents, stride):
    """Ranges (s, e) of at most seq_sents consecutive sentences that cover 0..n-1; consecutive ranges start `stride` apart; the last
    range is aligned to the end of the document."""
    if n <= 0:
        return []
    if n <= seq_sents:
        return [(0, n)]
    starts = list(range(0, n - seq_sents + 1, max(1, stride)))
    if starts[-1] + seq_sents < n:
        starts.append(n - seq_sents)
    return [(s, s + seq_sents) for s in starts]


def pack_groups(lens, window, together):
    """Indices of the sentences of a unit, grouped into backbone rows of at most `window` tokens (a sentence is never split)."""
    if not together:
        return [[i] for i in range(len(lens))]
    groups, cur, tot = [], [], 0
    for i, l in enumerate(lens):
        if cur and tot + l > window:
            groups.append(cur); cur, tot = [], 0
        cur.append(i); tot += l
    if cur:
        groups.append(cur)
    return groups


def collate(docs, units, C, with_labels=True, pad_id=0):
    """units: list of (doc_index, s, e). Builds the tensors for one batch:
       ids/mask (R, T): backbone rows (right padded); gsid (R, T): global sentence index of every token (-1 = pad);
       last_rows/last_pos (S,): position of every sentence's last token; unit_sizes (U,), unit_mask (U, Lmax),
       labels (U, Lmax) (IGNORE for padding), kd (U, Lmax, K) if the documents carry teacher scores."""
    window, together = C["WINDOW"], C["PACK"] == "together"
    rows_ids, rows_sid, last_rows, last_pos, sizes = [], [], [], [], []
    g0 = 0
    for di, s, e in units:
        sents = docs[di]["sents"][s:e]
        for grp in pack_groups([len(x) for x in sents], window, together):
            ids, sid = [], []
            for i in grp:
                ids += sents[i]; sid += [g0 + i] * len(sents[i])
                last_rows.append(len(rows_ids)); last_pos.append(len(ids) - 1)
            rows_ids.append(ids); rows_sid.append(sid)
        sizes.append(e - s); g0 += e - s
    R, T = len(rows_ids), max(len(r) for r in rows_ids)
    ids = torch.full((R, T), pad_id, dtype=torch.long)
    mask = torch.zeros((R, T), dtype=torch.bool)
    gsid = torch.full((R, T), -1, dtype=torch.long)
    for r, (a, b) in enumerate(zip(rows_ids, rows_sid)):
        ids[r, :len(a)] = torch.tensor(a); mask[r, :len(a)] = True; gsid[r, :len(b)] = torch.tensor(b)
    U, L = len(units), max(sizes)
    unit_mask = torch.arange(L).unsqueeze(0) < torch.tensor(sizes).unsqueeze(1)
    b = dict(ids=ids, mask=mask, gsid=gsid, last_rows=torch.tensor(last_rows), last_pos=torch.tensor(last_pos),
             n_sents=g0, unit_sizes=sizes, unit_mask=unit_mask)
    if with_labels and all(docs[di]["y"] is not None for di, _, _ in units):
        lab = torch.full((U, L), IGNORE, dtype=torch.long)
        for u, (di, s, e) in enumerate(units):
            lab[u, :e - s] = torch.tensor(docs[di]["y"][s:e])
        b["labels"] = lab
    if with_labels and all(docs[di].get("kd") is not None for di, _, _ in units):
        K = docs[units[0][0]]["kd"].shape[-1]
        kd = torch.zeros((U, L, K))
        for u, (di, s, e) in enumerate(units):
            kd[u, :e - s] = torch.as_tensor(docs[di]["kd"][s:e]).float()
        b["kd"] = kd
    return b


def to_device(b, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}


# ----------------------------------------------------------------------------------------------------------------------
# 3. Model = backbone + sentence pooling + head
# ----------------------------------------------------------------------------------------------------------------------
def pool_sentences(h, b, mode):
    """h: (R, T, d) backbone states -> (S, d_pool): mean of the sentence's tokens and/or its last token (a causal model has read the
    whole sentence at its last token)."""
    S, parts = b["n_sents"], []
    if mode in ("mean", "mean+last"):
        valid = b["gsid"] >= 0
        hv, sid = h[valid].float(), b["gsid"][valid]
        sums = torch.zeros(S, hv.shape[1], device=h.device).index_add_(0, sid, hv)
        cnt = torch.zeros(S, device=h.device).index_add_(0, sid, torch.ones(sid.shape[0], device=h.device))
        parts.append(sums / cnt.clamp(min=1).unsqueeze(1))
    if mode in ("last", "mean+last"):
        parts.append(h[b["last_rows"], b["last_pos"]].float())
    assert parts, f"unknown POOL {mode}"
    return torch.cat(parts, -1)


class BiLSTMLayer(nn.Module):
    """Bidirectional LSTM over the sentence vectors of a unit (padding packed away, so the backward direction never reads pads)."""

    def __init__(self, d_in, hidden, layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(d_in, hidden, num_layers=layers, batch_first=True, bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        self.out_dim = 2 * hidden

    def forward(self, x, mask):
        lengths = mask.sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x.float(), lengths, batch_first=True, enforce_sorted=False)
        with torch.autocast(device_type=x.device.type, enabled=False):
            out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.shape[1])
        return out


class SentenceHead(nn.Module):
    def __init__(self, d_in, n_labels, head_type="bilstm", lstm_hidden=256, lstm_layers=1, dropout=0.1, norm=True):
        super().__init__()
        self.norm = nn.LayerNorm(d_in) if norm else nn.Identity()
        self.drop = nn.Dropout(dropout)
        self.lstm = None
        d_out = d_in
        if head_type == "bilstm":
            self.lstm = BiLSTMLayer(d_in, lstm_hidden, lstm_layers, dropout)
            d_out = self.lstm.out_dim
        elif head_type != "linear":
            raise ValueError(head_type)
        self.out = nn.Linear(d_out, n_labels)

    def forward(self, v, mask):
        x = self.drop(self.norm(v))
        if self.lstm is not None:
            x = self.lstm(x, mask)
        return self.out(x)


class RRModel(nn.Module):
    """Same attribute names as the Phase-2 NERModel (backbone, soft_prompt, head_parameters) so the technique code can treat both alike."""

    def __init__(self, backbone, d_model, n_labels, C):
        super().__init__()
        self.backbone = backbone
        self.soft_prompt = None
        self.pool = C["POOL"]
        d_pool = d_model * (2 if self.pool == "mean+last" else 1)
        self.sent_head = SentenceHead(d_pool, n_labels, C["HEAD_TYPE"], C["HEAD_LSTM_HIDDEN"], 1, C["HEAD_DROPOUT"], C["HEAD_NORM"])
        self.n_labels = n_labels

    def head_parameters(self):
        return list(self.sent_head.parameters())

    def forward(self, b):
        h = self.backbone.features(b["ids"], b["mask"], self.soft_prompt)
        v = pool_sentences(h, b, self.pool)                                  # (S, d_pool), units in order
        um = b["unit_mask"]
        vp = v.new_zeros(um.shape[0], um.shape[1], v.shape[1])
        vp[um] = v                                                           # row-major order of True entries == global sentence order
        return self.sent_head(vp, um)                                        # (U, Lmax, n_labels)


# ----------------------------------------------------------------------------------------------------------------------
# 4. Loss, class weights, training
# ----------------------------------------------------------------------------------------------------------------------
def class_weights(train_docs, n_labels, scheme, cap):
    if scheme == "none":
        return None
    cnt = torch.zeros(n_labels)
    for d in train_docs:
        for y in d["y"]:
            cnt[y] += 1
    present = cnt > 0
    n = cnt.sum()
    if scheme == "inv_sqrt":
        w = torch.sqrt(n / (present.sum() * cnt.clamp(min=1)))
    elif scheme == "inv":
        w = n / (present.sum() * cnt.clamp(min=1))
    else:
        raise ValueError(scheme)
    w = torch.where(present, w.clamp(max=cap), torch.zeros_like(w))
    return w / w[present].mean()                                           # mean weight over the labels present = 1


def sentence_loss(logits, labels, cw, C):
    K = logits.shape[-1]
    return F.cross_entropy(logits.reshape(-1, K).float(), labels.reshape(-1), weight=cw, ignore_index=IGNORE,
                           label_smoothing=C.get("LABEL_SMOOTHING", 0.0))


def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def trainable_state(model):
    """Everything that can differ from the Phase-1 weights: trainable parameters + routing buffers that fine-tuning may update."""
    st = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
    for n, bf in model.named_buffers():
        if n.endswith(("select_bias", ".tau", ".tau_ready", ".ada_mask")):
            st[n] = bf.detach().to("cpu", copy=True)
    return st


def load_trainable_state(model, st):
    known = dict(model.named_parameters()); known.update(dict(model.named_buffers()))
    missing = [k for k in st if k not in known]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    model.load_state_dict(st, strict=False)


def train_rr(model, train_docs, C, device, eval_fn=None, epochs=None, log=print, keep_best=True):
    """Trains the trainable parameters on the units of train_docs. eval_fn(model) -> (score, details) is called after every epoch; the
    best epoch's state is kept. Optional model hooks (used by AdaLoRA): extra_loss(), on_before_step(step, total)."""
    epochs = epochs or C["EPOCHS"]
    model.to(device)
    torch.manual_seed(C["SEED"]); random.seed(C["SEED"])
    items = [(di, s, e) for di, d in enumerate(train_docs) for (s, e) in make_units(d["n"], C["SEQ_SENTS"], C["TRAIN_STRIDE"])]
    cw = class_weights(train_docs, model.n_labels, C["CLASS_WEIGHTING"], C["CW_CAP"])
    cw = cw.to(device) if cw is not None else None
    head_params = [p for p in model.head_parameters() if p.requires_grad]
    hp = set(id(p) for p in head_params)
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in hp]
    groups = [dict(params=head_params, lr=C["HEAD_LR"])]
    if other:
        groups.append(dict(params=other, lr=C["LR"]))
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    steps_per_epoch = math.ceil(len(items) / (C["BATCH"] * C["ACCUM"]))
    total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    hist, best, best_state, best_ep, bad = [], -1.0, None, 0, 0
    t0, tokens, gstep = time.time(), 0, 0
    kd_on = C.get("KD_ALPHA", 0) > 0 and all(d.get("kd") is not None for d in train_docs)
    log(f"training: {len(items)} units/epoch, {steps_per_epoch} optimizer steps/epoch, {epochs} epochs" + (", KD on" if kd_on else ""))
    for ep in range(1, epochs + 1):
        model.train()
        order = items[:]; random.Random(C["SEED"] * 1000 + ep).shuffle(order)
        run_loss, nb, skipped, pending = 0.0, 0, 0, 0
        opt.zero_grad(set_to_none=True)
        for bi in range(0, len(order), C["BATCH"]):
            b = to_device(collate(train_docs, order[bi:bi + C["BATCH"]], C), device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(b)
            logits = logits.float()
            loss = sentence_loss(logits, b["labels"], cw, C)
            if kd_on:
                T = C["KD_TEMPERATURE"]; um = b["unit_mask"]
                kd = F.kl_div(F.log_softmax(logits[um] / T, -1), F.softmax(b["kd"][um] / T, -1), reduction="batchmean") * T * T
                loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd
            if hasattr(model, "extra_loss"):
                loss = loss + model.extra_loss()
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss / C["ACCUM"]).backward()
            nb += 1; pending += 1; tokens += int(b["mask"].sum()); run_loss += loss.item()
            if pending >= C["ACCUM"] or (bi + C["BATCH"] >= len(order) and pending > 0):
                scaler.unscale_(opt)
                if hasattr(model, "on_before_step"):
                    model.on_before_step(gstep, total)
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], C["GRAD_CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); gstep += 1
                opt.zero_grad(set_to_none=True); pending = 0
        if pending > 0:                                   # gradients left over (the last batch was non-finite): use them, never drop them
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], C["GRAD_CLIP"])
            scaler.step(opt); scaler.update(); sched.step(); gstep += 1
            opt.zero_grad(set_to_none=True); pending = 0
        row = dict(epoch=ep, train_loss=round(run_loss / max(1, nb), 5), seconds=round(time.time() - t0, 1))
        if skipped:
            row["skipped_nonfinite_batches"] = skipped
        if eval_fn is not None:
            score, det = eval_fn(model)
            row.update(heldout_macroF1=round(score, 5), **det)
            if score > best:
                best, best_ep, best_state, bad = score, ep, (trainable_state(model) if keep_best else None), 0
            else:
                bad += 1
        hist.append(row)
        log("  epoch %d: " % ep + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if nb == 0:
            log("  every batch of this epoch was non-finite: stopping"); break
        if eval_fn is not None and C["PATIENCE"] and bad >= C["PATIENCE"]:
            log(f"  early stop: no improvement for {bad} epochs"); break
    if eval_fn is None:
        best_state, best_ep = trainable_state(model), len(hist)
    secs = time.time() - t0
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(history=hist, best_epoch=best_ep, best_score=best, best_state=best_state, train_seconds=round(secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, tokens_per_sec=round(tokens / max(secs, 1e-9), 1))


# ----------------------------------------------------------------------------------------------------------------------
# 5. Inference and decoding
# ----------------------------------------------------------------------------------------------------------------------
@torch.no_grad()
def predict_logprobs(model, docs, C, device):
    """Per document a (n_sentences, K) tensor of log-probabilities. Sentences covered by several overlapping units are merged:
    MERGE='center' keeps the unit in which the sentence is farthest from the unit's edges, 'average' averages the log-probabilities."""
    model.eval(); model.to(device)
    K = model.n_labels
    units = [(di, s, e) for di, d in enumerate(docs) for (s, e) in make_units(d["n"], C["SEQ_SENTS"], C["INFER_STRIDE"])]
    units.sort(key=lambda u: sum(len(x) for x in docs[u[0]]["sents"][u[1]:u[2]]))      # similar lengths together: less padding
    acc = [torch.zeros(d["n"], K) for d in docs]
    best = [torch.full((d["n"],), -1.0) for d in docs]
    cnt = [torch.zeros(d["n"]) for d in docs]
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    for bi in range(0, len(units), C["INFER_BATCH"]):
        chunk = units[bi:bi + C["INFER_BATCH"]]
        b = to_device(collate(docs, chunk, C, with_labels=False), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = model(b)
        lp = F.log_softmax(lg.float(), -1).cpu()
        for u, (di, s, e) in enumerate(chunk):
            L = e - s
            pos = torch.arange(L)
            if C["MERGE"] == "average":
                acc[di][s:e] += lp[u, :L]; cnt[di][s:e] += 1
            else:
                score = torch.minimum(pos, L - 1 - pos).float()
                upd = score > best[di][s:e]
                acc[di][s:e][upd] = lp[u, :L][upd]
                best[di][s:e] = torch.where(upd, score, best[di][s:e])
    if C["MERGE"] == "average":
        acc = [a / c.clamp(min=1).unsqueeze(1) for a, c in zip(acc, cnt)]
    return acc


def learn_logT(train_docs, K, smoothing=1.0):
    """Label-transition log-probabilities log P(y_t | y_{t-1}) from the training documents (add-`smoothing`)."""
    cnt = torch.full((K, K), float(smoothing))
    for d in train_docs:
        for a, b in zip(d["y"][:-1], d["y"][1:]):
            cnt[a, b] += 1
    return (cnt / cnt.sum(1, keepdim=True)).log()


def decode_ids(lp, logT=None, lam=0.0):
    """Label sequence for one document: per-sentence argmax, or Viterbi over lp + lam * logT (smooths isolated label flips)."""
    n = lp.shape[0]
    if n == 0:
        return []
    if logT is None or lam == 0.0:
        return lp.argmax(-1).tolist()
    K = lp.shape[1]
    T = lam * logT
    score, back = lp[0].clone(), torch.zeros(n, K, dtype=torch.long)
    for t in range(1, n):
        s, arg = (score.unsqueeze(1) + T).max(0)
        back[t] = arg; score = s + lp[t]
    path = [int(score.argmax())]
    for t in range(n - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]


# ----------------------------------------------------------------------------------------------------------------------
# 6. Scoring (macro-F1 overall and per domain)
# ----------------------------------------------------------------------------------------------------------------------
def macro_scores(gold, pred, labels):
    """gold/pred: flat lists of label ids. macro_F1 averages over the labels that occur in gold OR pred (scikit-learn's default);
    macro_F1_gold over the labels in gold; macro_F1_all13 over all labels. The official aggregation is not known until a submission."""
    K = len(labels)
    tp, fp, fn = [0] * K, [0] * K, [0] * K
    for g, p in zip(gold, pred):
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1; fn[g] += 1
    f1, P, R, per = [0.0] * K, [0.0] * K, [0.0] * K, {}
    for c in range(K):
        P[c] = tp[c] / (tp[c] + fp[c]) if tp[c] + fp[c] else 0.0
        R[c] = tp[c] / (tp[c] + fn[c]) if tp[c] + fn[c] else 0.0
        f1[c] = 2 * tp[c] / (2 * tp[c] + fp[c] + fn[c]) if 2 * tp[c] + fp[c] + fn[c] else 0.0
        per[labels[c]] = dict(P=round(P[c], 4), R=round(R[c], 4), F1=round(f1[c], 4), support=tp[c] + fn[c], predicted=tp[c] + fp[c])
    union = [c for c in range(K) if tp[c] + fp[c] + fn[c] > 0]
    gold_present = [c for c in range(K) if tp[c] + fn[c] > 0]
    mean = lambda xs, idx: sum(xs[i] for i in idx) / max(1, len(idx))
    return dict(macro_F1=mean(f1, union), macro_P=mean(P, union), macro_R=mean(R, union),
                macro_F1_gold=mean(f1, gold_present), macro_F1_all13=mean(f1, range(K)),
                accuracy=sum(tp) / max(1, len(gold)), per_label=per, n=len(gold))


def score_docs(gold, pred, domain_of, labels):
    """gold/pred: {doc_id: [label ids]}. Returns the scores overall, and for each domain present."""
    out = {}
    for name, keep in [("all", lambda d: True)] + [(dom, (lambda d, dom=dom: domain_of[d] == dom)) for dom in sorted(set(domain_of.values()))]:
        ids = [d for d in gold if keep(d)]
        out[name] = macro_scores([y for d in ids for y in gold[d]], [y for d in ids for y in pred[d]], labels)
    return out


def confusion_counts(gold, pred, labels, top=15):
    c = Counter((labels[g], labels[p]) for d in gold for g, p in zip(gold[d], pred[d]) if g != p)
    return [(f"{a}->{b}", n) for (a, b), n in c.most_common(top)]


def eval_docs(model, docs, C, device, logT=None):
    """(score, details) for the training loop: macro-F1 over all held-out sentences plus the per-domain scores."""
    lps = predict_logprobs(model, docs, C, device)
    labels = C["LABELS"]
    gold = {d["id"]: d["y"] for d in docs}
    pred = {d["id"]: decode_ids(lp, logT, C.get("TRANS_LAMBDA", 0.0)) for d, lp in zip(docs, lps)}
    dom = {d["id"]: d["domain"] for d in docs}
    S = score_docs(gold, pred, dom, labels)
    det = {f"F1_{k}": round(v["macro_F1"], 4) for k, v in S.items() if k != "all"}
    det["acc"] = round(S["all"]["accuracy"], 4)
    return S["all"]["macro_F1"], det


# ----------------------------------------------------------------------------------------------------------------------
# 7. Cross-validation folds, runs, submission
# ----------------------------------------------------------------------------------------------------------------------
def make_folds(docs, k, seed=0):
    """Document-level folds, stratified by domain, balanced by number of sentences (largest documents first)."""
    folds = [[] for _ in range(k)]
    size = [0] * k
    by_dom = defaultdict(list)
    for i, d in enumerate(docs):
        by_dom[d["domain"]].append(i)
    rng = random.Random(seed)
    for dom in sorted(by_dom):
        idx = by_dom[dom][:]
        rng.shuffle(idx)
        idx.sort(key=lambda i: -docs[i]["n"])
        dsize = [0] * k
        for i in idx:
            f = min(range(k), key=lambda j: (dsize[j], size[j], j))
            folds[f].append(i); dsize[f] += docs[i]["n"]; size[f] += docs[i]["n"]
    return folds


def fold_report(docs, folds, labels):
    rows = []
    for f, idx in enumerate(folds):
        c = Counter(y for i in idx for y in docs[i]["y"])
        rows.append(dict(fold=f, docs=len(idx), sentences=sum(docs[i]["n"] for i in idx), domains=dict(Counter(docs[i]["domain"] for i in idx)),
                         labels={labels[k]: c[k] for k in range(len(labels)) if c[k]}))
    return rows


def run_cv(build_fn, docs, folds, C, device, fold_ids=None, test_docs=None, log=print):
    """For every fold: train on the other folds, pick the best epoch on the held-out fold, keep the held-out log-probabilities and (optionally)
    the test log-probabilities of that fold model. build_fn() -> (model, info). Returns dict(per_fold, curves, heldout_lp, test_lp)."""
    fold_ids = list(range(len(folds))) if fold_ids is None else fold_ids
    out = dict(per_fold={}, curves={}, heldout_lp={}, test_lp={})
    for k in fold_ids:
        va = [docs[i] for i in folds[k]]
        tr = [docs[i] for f in range(len(folds)) if f != k for i in folds[f]]
        model, info = build_fn()
        logT = learn_logT(tr, model.n_labels)
        log(f"== fold {k}: train {len(tr)} docs / held-out {len(va)} docs ==")
        res = train_rr(model, tr, C, device, eval_fn=lambda m: eval_docs(m, va, C, device, logT), log=log)
        load_trainable_state(model, res["best_state"])
        out["heldout_lp"][k] = {d["id"]: lp for d, lp in zip(va, predict_logprobs(model, va, C, device))}
        if test_docs is not None:
            out["test_lp"][k] = {d["id"]: lp for d, lp in zip(test_docs, predict_logprobs(model, test_docs, C, device))}
        out["per_fold"][k] = {kk: v for kk, v in res.items() if kk != "best_state"}
        out["per_fold"][k]["technique_info"] = info
        out["curves"][k] = [r.get("heldout_macroF1") for r in res["history"]]
        del model, res
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return out


def choose_epoch(curves, smooth=3):
    """Epoch (1-based) with the best mean held-out score across folds, using a `smooth`-epoch moving average so a flat plateau
    wins over a single noisy peak."""
    n = min(len(c) for c in curves.values())
    mean = [sum(c[i] for c in curves.values()) / len(curves) for i in range(n)]
    sm = [sum(mean[max(0, i - smooth // 2): i + smooth // 2 + 1]) / len(mean[max(0, i - smooth // 2): i + smooth // 2 + 1]) for i in range(n)]
    return int(max(range(n), key=lambda i: sm[i])) + 1, mean


def validate_submission(pred, test_raw, labels):
    """pred: {doc_id: [label strings]}; test_raw: {doc_id: [sentences]}. Raises on any format problem."""
    assert set(pred) == set(test_raw), f"id mismatch: missing {len(set(test_raw) - set(pred))}, extra {len(set(pred) - set(test_raw))}"
    for did, sents in test_raw.items():
        p = pred[did]
        assert isinstance(p, list) and len(p) == len(sents), f"{did}: {len(p)} predictions for {len(sents)} sentences"
        assert all(isinstance(x, str) and x in labels for x in p), f"{did}: a prediction is not an official label string"
    return True


def write_submission(pred_ids, labels, test_raw, out_dir, name):
    """pred_ids: {doc_id: [label ids]} -> prediction.json + a zip with prediction.json at its root. Returns the paths."""
    pred = {did: [labels[i] for i in ids] for did, ids in pred_ids.items()}
    validate_submission(pred, test_raw, labels)
    os.makedirs(out_dir, exist_ok=True)
    jpath = os.path.join(out_dir, f"{name}_prediction.json")
    text = json.dumps(pred, ensure_ascii=False)
    assert "null" not in text, "a null label slipped into the submission"
    with open(jpath, "w", encoding="utf-8") as f:
        f.write(text)
    zpath = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jpath, arcname="prediction.json")
    return jpath, zpath

## 4. Phase-3 data (gold labels)

In [4]:
P3 = "Phase_3_RR"
def load_json(p):
    with open(p, encoding="utf-8") as f:          # plain json: the label string "None" must stay a string
        return json.load(f)

LABELS = labels_from_vocab(load_json(get_dataset_file(f"{P3}/Train/label_vocab.json")))
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
N_LABELS = len(LABELS)
LABELLED_RAW, TEST_RAW = [], {}
for dom in ("CL", "IT"):
    for sp in ("train", "dev"):
        ds = as_doc_list(load_json(get_dataset_file(f"{P3}/Train/{dom}_{sp}.json")), dom)
        for d in ds:
            d["split"] = sp
        LABELLED_RAW += ds
    TEST_RAW[dom] = load_json(get_dataset_file(f"{P3}/Test/{dom}_test.json"))
TEST_ALL = {k: v for dom in TEST_RAW for k, v in TEST_RAW[dom].items()}
assert len(TEST_ALL) == sum(len(v) for v in TEST_RAW.values()), "a test document id occurs in both domains"
assert len({d["id"] for d in LABELLED_RAW}) == len(LABELLED_RAW), "duplicate labelled ids"
assert not ({d["id"] for d in LABELLED_RAW} & set(TEST_ALL)), "a test id is also a labelled id"
for d in LABELLED_RAW:
    assert len(d["text"]) == len(d["labels"]) and all(l in LABEL2ID for l in d["labels"])
if QUICK_TEST:                                   # tiny, fast check: 4 documents per domain, first 40 sentences each, 1 test document per domain
    keep = []
    for dom in ("CL", "IT"):
        for d in sorted([x for x in LABELLED_RAW if x["domain"] == dom], key=lambda x: len(x["text"]))[:4]:
            d["text"], d["labels"] = d["text"][:40], d["labels"][:40]; keep.append(d)
    LABELLED_RAW = keep
    TEST_RAW = {dom: {k: v[:30] for k, v in list(TEST_RAW[dom].items())[:1]} for dom in TEST_RAW}
    TEST_ALL = {k: v for dom in TEST_RAW for k, v in TEST_RAW[dom].items()}
print("labels:", LABELS)
print("labelled documents:", Counter((d["domain"], d["split"]) for d in LABELLED_RAW))
print("test documents:", {dom: (len(v), sum(len(x) for x in v.values())) for dom, v in TEST_RAW.items()}, "(documents, sentences)")

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
labelled documents: Counter({('CL', 'train'): 40, ('IT', 'train'): 40, ('CL', 'dev'): 5, ('IT', 'dev'): 5})
test documents: {'CL': (5, 1339), 'IT': (5, 858)} (documents, sentences)


## 5. Ensemble, bias calibration, test predictions, error analysis

In [5]:
# ======================================================================================================================
# 1. Load the labelled documents (gold labels, domains) -- no model is needed in this notebook
# ======================================================================================================================
LABELLED = [dict(id=d["id"], domain=d["domain"], y=[LABEL2ID[l] for l in d["labels"]], n=len(d["labels"])) for d in LABELLED_RAW]
GOLD = {d["id"]: d["y"] for d in LABELLED}
DOM = {d["id"]: d["domain"] for d in LABELLED}
RES_DIR = f"{WORK}/ens_files"; os.makedirs(RES_DIR, exist_ok=True)
SUB_DIR = f"{WORK}/p3_submissions_{SWEEP_NAME}"; os.makedirs(SUB_DIR, exist_ok=True)


def _get(name):
    try:
        return fetch_model_file(name, f"{RES_DIR}/{name}")
    except Exception:
        return None


def load_oof(prefix, setup):
    """Out-of-fold log-probabilities {doc_id: (n, 13) tensor} at the epoch chosen for this setup, and {doc_id: fold}. Handles pooled ('all')
    and separate (CL + IT) setups."""
    for groups in (["all"], ["CL", "IT"]):
        oof, fold_of, ok = {}, {}, True
        for g in groups:
            files = [_get(f"p3res_{prefix}__{setup}__{g}__f{k}.pt") for k in range(K_FOLDS)]
            if any(f is None for f in files):
                ok = False; break
            rs = [torch.load(f, map_location="cpu", weights_only=False) for f in files]
            E, _ = choose_epoch({k: r["curve"] for k, r in enumerate(rs)})
            E = min(E, min(len(r["curve"]) for r in rs))
            for k, r in enumerate(rs):
                for did, lp in r["lps"][E].items():
                    oof[did] = lp.float(); fold_of[did] = k
        if ok:
            return oof, fold_of
    return None, None


def load_test(fname, name):
    p = _get(fname)
    if p is None:
        return None
    d = torch.load(p, map_location="cpu", weights_only=False)
    return d.get(name)


OOF, TEST, FOLD_OF = {}, {}, None
for label, prefix, setup, tfile, tname in MEMBERS:
    oof, fo = load_oof(prefix, setup)
    tl = load_test(tfile, tname)
    if oof is None or len(oof) != len(LABELLED):
        print(f"skipped {label}: out-of-fold files not found"); continue
    OOF[label] = oof
    if FOLD_OF is None:
        FOLD_OF = fo
    if tl is not None:
        TEST[label] = tl
    else:
        print(f"note: {label} has out-of-fold predictions but no test log-probabilities ({tfile}:{tname}); it cannot be part of the test ensemble")
print("members loaded:", list(OOF), "| with test predictions:", list(TEST))
IDS = [d["id"] for d in LABELLED]
Y = np.concatenate([GOLD[i] for i in IDS])
DOMS = np.concatenate([[DOM[i]] * len(GOLD[i]) for i in IDS])
FOLDS_ARR = np.concatenate([[FOLD_OF[i]] * len(GOLD[i]) for i in IDS])
K = N_LABELS


def stack(members, ids=IDS):
    return np.mean([np.concatenate([OOF[m][i].numpy() for i in ids]) for m in members], axis=0)


def macro_f1_np(y, p):
    cm_tp = np.bincount(y[y == p], minlength=K); n_g = np.bincount(y, minlength=K); n_p = np.bincount(p, minlength=K)
    denom = n_g + n_p
    f1 = np.where(denom > 0, 2 * cm_tp / np.maximum(denom, 1), 0.0)
    present = denom > 0
    return float(f1[present].mean())


def report(name, X, bias=None):
    P = (X + (0 if bias is None else bias)).argmax(1)
    f_all = macro_f1_np(Y, P)
    f_dom = {d: macro_f1_np(Y[DOMS == d], P[DOMS == d]) for d in ("CL", "IT")}
    print(f"{name:42s} macro-F1 {f_all:.4f}  CL {f_dom['CL']:.4f}  IT {f_dom['IT']:.4f}")
    return f_all


# ======================================================================================================================
# 2. Single members, simple averages, greedy selection (all on out-of-fold predictions)
# ======================================================================================================================
print("\n== single members (out-of-fold; the same numbers as the sweep tables) ==")
single = {m: report(m, stack([m])) for m in OOF}
print("\n== averages of log-probabilities ==")
members_all = list(OOF)
f_avg_all = report(f"average of all {len(members_all)} members", stack(members_all))
top3 = sorted(single, key=single.get, reverse=True)[:3]
f_top3 = report(f"average of the top 3 ({', '.join(top3)})", stack(top3))
chosen = [max(single, key=single.get)]
best = single[chosen[0]]
while True:
    cand = [(report_f, m) for m in OOF if m not in chosen for report_f in [macro_f1_np(Y, stack(chosen + [m]).argmax(1))]]
    if not cand:
        break
    f, m = max(cand)
    if f > best + MIN_GAIN:
        chosen.append(m); best = f
    else:
        break
print(f"\ngreedy selection (adds a member only if it gains > {MIN_GAIN}): {chosen} -> {best:.4f}")
ENS_MEMBERS = [m for m in chosen if m in TEST] or [m for m in top3 if m in TEST] or list(TEST)
print("members used for the test ensemble (need test predictions):", ENS_MEMBERS)
X_ens = stack(ENS_MEMBERS)
f_ens = report("ensemble used for the submission", X_ens)


# ======================================================================================================================
# 3. Per-label bias calibration (macro-F1 rewards finding rare labels such as Dissent; an additive offset on the log-probability of a label
#    changes how readily it is predicted). The honest estimate learns the offsets on 4 folds and applies them to the 5th.
# ======================================================================================================================
def learn_bias(X, y):
    b = np.zeros(K)
    cur = macro_f1_np(y, (X + b).argmax(1))
    for _ in range(BIAS_ROUNDS):
        for c in range(K):
            best_d, best_f = b[c], cur
            for d in BIAS_GRID:
                b2 = b.copy(); b2[c] = d
                f = macro_f1_np(y, (X + b2).argmax(1))
                if f > best_f + 1e-4:
                    best_d, best_f = d, f
            b[c] = best_d; cur = best_f
    return b, cur


P_cv = np.zeros(len(Y), dtype=int)
for k in range(K_FOLDS):
    tr, te = FOLDS_ARR != k, FOLDS_ARR == k
    b_k, _ = learn_bias(X_ens[tr], Y[tr])
    P_cv[te] = (X_ens[te] + b_k).argmax(1)
f_bias_honest = macro_f1_np(Y, P_cv)
print(f"\nbias calibration, honest estimate (offsets learned on 4 folds, applied to the 5th): {f_bias_honest:.4f}  (no bias: {f_ens:.4f}, change {f_bias_honest - f_ens:+.4f})")
BIAS, f_bias_fit = learn_bias(X_ens, Y)
print(f"offsets learned on all folds (fit score {f_bias_fit:.4f}):", {LABELS[c]: float(BIAS[c]) for c in range(K) if abs(BIAS[c]) > 1e-9})
USE_BIAS = (f_bias_honest - f_ens) > 0.003
print("bias will be applied in the second submission:", USE_BIAS, "(it is applied only if the honest estimate gains more than 0.003)")


# ======================================================================================================================
# 4. Test predictions: average the members' test log-probabilities, optionally add the offsets, write the submission zips
# ======================================================================================================================
def test_pred(members, bias):
    pred = {}
    for did in TEST_ALL:
        X = np.mean([TEST[m][did].numpy() for m in members], axis=0)
        pred[did] = (X + bias).argmax(1).tolist()
    return pred


for tag, bias in [("ens", np.zeros(K)), ("ens_bias", BIAS)]:
    if tag == "ens_bias" and not USE_BIAS:
        print("skipping ens_bias (no honest gain)"); continue
    if not all(m in TEST for m in ENS_MEMBERS):
        print("missing test predictions, no submission"); break
    pred = test_pred(ENS_MEMBERS, bias)
    jpath, zpath = write_submission(pred, LABELS, TEST_ALL, SUB_DIR, f"{SWEEP_NAME}_{tag}")
    dist = Counter(LABELS[i] for ids in pred.values() for i in ids)
    print(f"wrote {zpath}; predicted label counts: {dict(dist.most_common())}")
    upload_model_file(zpath)


# ======================================================================================================================
# 5. Error analysis on the out-of-fold ensemble: CL vs IT per label, closely related roles, Dissent
# ======================================================================================================================
P_fin = (X_ens + (BIAS if USE_BIAS else 0)).argmax(1)
print("\n== per-label F1 of the final out-of-fold ensemble: all / CL / IT (support all / CL / IT) ==")
ANALYSIS = {}
for c, l in enumerate(LABELS):
    row = {}
    for nm, msk in [("all", np.ones(len(Y), bool)), ("CL", DOMS == "CL"), ("IT", DOMS == "IT")]:
        y, p = Y[msk], P_fin[msk]
        tp = int(((y == c) & (p == c)).sum()); fp = int(((y != c) & (p == c)).sum()); fn = int(((y == c) & (p != c)).sum())
        row[nm] = dict(F1=round(2 * tp / max(1, 2 * tp + fp + fn), 4), P=round(tp / max(1, tp + fp), 4), R=round(tp / max(1, tp + fn), 4), support=tp + fn)
    ANALYSIS[l] = row
    print(f"  {l:24s} F1 {row['all']['F1']:.3f} / {row['CL']['F1']:.3f} / {row['IT']['F1']:.3f}   ({row['all']['support']} / {row['CL']['support']} / {row['IT']['support']})")

GROUPS = {"petitioner vs respondent arguments": ["ArgumentPetitioner", "ArgumentRespondent"],
          "precedent roles": ["PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled"],
          "rulings and reasoning": ["RulingByLowerCourt", "RulingByPresentCourt", "RatioOfTheDecision"],
          "facts, issues, statutes": ["Fact", "Issue", "Statute"]}
CONF = {}
for gname, labs in GROUPS.items():
    ids = [LABEL2ID[l] for l in labs]
    print(f"\n== {gname}: rows = gold label, columns = predicted label (last column = predicted as any other label) ==")
    print(" " * 26 + "".join(f"{l[:12]:>14s}" for l in labs) + f"{'other':>14s}")
    CONF[gname] = {}
    for gl, gi in zip(labs, ids):
        m = Y == gi
        cnts = [int((P_fin[m] == pj).sum()) for pj in ids]
        other = int(m.sum()) - sum(cnts)
        CONF[gname][gl] = cnts + [other]
        print(f"  {gl:24s}" + "".join(f"{x:14d}" for x in cnts) + f"{other:14d}")

print("\n== Dissent ==")
for dom in ("CL", "IT"):
    docs = [i for i in IDS if DOM[i] == dom]
    with_d = [i for i in docs if LABEL2ID["Dissent"] in GOLD[i]]
    n_d = sum(GOLD[i].count(LABEL2ID["Dissent"]) for i in docs)
    print(f"  {dom}: {len(with_d)} of {len(docs)} documents contain Dissent ({n_d} sentences)")
print("  predicted Dissent sentences (out-of-fold ensemble):", int((P_fin == LABEL2ID["Dissent"]).sum()), "| gold:", int((Y == LABEL2ID["Dissent"]).sum()))
print("  PrecedentOverruled gold sentences:", int((Y == LABEL2ID["PrecedentOverruled"]).sum()), "| predicted:", int((P_fin == LABEL2ID["PrecedentOverruled"]).sum()))


# ======================================================================================================================
# 6. Every member on the closely related roles (how each technique separates them), out-of-fold
# ======================================================================================================================
def f1_of(y, p, c):
    tp = int(((y == c) & (p == c)).sum()); fp = int(((y != c) & (p == c)).sum()); fn = int(((y == c) & (p != c)).sum())
    return 2 * tp / max(1, 2 * tp + fp + fn)


def rate(y, p, a, b):
    m = y == LABEL2ID[a]
    return float((p[m] == LABEL2ID[b]).mean()) if m.any() else float("nan")


REL = {}
cols_f1 = ["ArgumentPetitioner", "ArgumentRespondent", "PrecedentReliedUpon", "PrecedentNotReliedUpon", "RulingByLowerCourt", "RulingByPresentCourt", "RatioOfTheDecision"]
cols_cf = [("ArgumentPetitioner", "ArgumentRespondent"), ("ArgumentRespondent", "ArgumentPetitioner"), ("PrecedentReliedUpon", "PrecedentNotReliedUpon"),
           ("PrecedentNotReliedUpon", "PrecedentReliedUpon"), ("PrecedentReliedUpon", "RatioOfTheDecision"), ("RatioOfTheDecision", "PrecedentReliedUpon"),
           ("RulingByLowerCourt", "RatioOfTheDecision")]
print("\n== F1 of the related roles per member (out-of-fold) ==")
print(f"{'member':24s}" + "".join(f"{c[:9]:>10s}" for c in cols_f1))
for m in OOF:
    p = stack([m]).argmax(1)
    REL[m] = dict(f1={c: round(f1_of(Y, p, LABEL2ID[c]), 3) for c in cols_f1}, confusion={f"{a}->{b}": round(rate(Y, p, a, b), 3) for a, b in cols_cf})
    print(f"{m:24s}" + "".join(f"{REL[m]['f1'][c]:10.3f}" for c in cols_f1))
print("\n== share of gold sentences confused with the related role (gold -> predicted), per member ==")
print(f"{'member':24s}" + "".join(f"{(a[:5] + '>' + b[:5]):>12s}" for a, b in cols_cf))
for m in OOF:
    print(f"{m:24s}" + "".join(f"{REL[m]['confusion'][f'{a}->{b}']:12.3f}" for a, b in cols_cf))


# ======================================================================================================================
# 7. Size and compute table (training; from the cached cross-validation files)
# ======================================================================================================================
EFF = []
print("\n== size and compute per setup (pooled; means over the 5 folds) ==")
print(f"{'setup':20s} {'total M':>8s} {'trainable M':>11s} {'stored MB':>9s} {'min/fold':>8s} {'peak GB':>8s} {'tokens/s':>9s}")
for label, prefix, setup in EFF_SETUPS:
    files = [_get(f"p3res_{prefix}__{setup}__all__f{k}.pt") for k in range(K_FOLDS)]
    if any(f is None for f in files):
        continue
    rs = [torch.load(f, map_location="cpu", weights_only=False) for f in files]
    inf = rs[0]["info"]
    row = dict(label=label, total_M=inf["params_total"] / 1e6, trainable_M=inf["params_trainable"] / 1e6, stored_MB=inf["model_size_mb"],
               min_per_fold=sum(r["seconds"] for r in rs) / len(rs) / 60, peak_GB=max((r["peak_mem_gb"] or 0) for r in rs),
               tokens_per_s=sum((r["tokens_per_sec"] or 0) for r in rs) / len(rs))
    EFF.append(row)
    print(f"{label:20s} {row['total_M']:8.1f} {row['trainable_M']:11.2f} {row['stored_MB']:9.0f} {row['min_per_fold']:8.1f} {row['peak_GB']:8.1f} {row['tokens_per_s']:9.0f}")

SUMMARY = dict(members=list(OOF), single=single, avg_all=f_avg_all, top3=top3, f_top3=f_top3, greedy=chosen, f_greedy=best, ensemble_members=ENS_MEMBERS,
               f_ens=f_ens, f_bias_honest=f_bias_honest, bias={LABELS[c]: float(BIAS[c]) for c in range(K)}, use_bias=bool(USE_BIAS),
               per_label=ANALYSIS, related_role_confusions=CONF, related_roles_per_member=REL, efficiency=EFF)
path = f"{WORK}/summary_{SWEEP_NAME}.json"
with open(path, "w") as f:
    json.dump(SUMMARY, f, indent=1, default=str)
upload_model_file(path)
for p in sorted(os.listdir(SUB_DIR)):
    print("  ", f"{SUB_DIR}/{p}", os.path.getsize(f"{SUB_DIR}/{p}") // 1024, "KB")
print(f"done. Send back: summary_{SWEEP_NAME}.json and the zips you want to submit.")


members loaded: ['moe_full_ft', 'moe_lora', 'moe_lorma', 'moe_ia3', 'moe_lora_seq48', 'moe_lora_win2048', 'moe_lorma_seq48', 'moe_full_ft_seq48', 'moe_rslora_seq48', 'moe_ia3_seq48', 'moe_pos_lora', 'moe_pos_lorma', 'moe_frozen_s48', 'moe_cw_inv', 'gqa_full_ft_seq48', 'gqa_qlora_seq48', 'moe_pos_dom_lora', 'moe_pos_cwinv_lora', 'moe_pos_lora_avg', 'moe_pos_lora_ls', 'moe_pos_lora_seed1', 'moe_pos_lora_seed2', 'moe_pos_ia3', 'moe_pos_frozen', 'moe_pos_dom_ia3', 'moe_pos_rslora', 'moe_pos_qlora', 'moe_pos_adalora', 'moe_pos_lora_s96', 'moe_pos_lora_s128', 'moe_pos_ia3_s96'] | with test predictions: ['moe_full_ft', 'moe_lora', 'moe_lorma', 'moe_ia3', 'moe_lora_seq48', 'moe_lora_win2048', 'moe_lorma_seq48', 'moe_full_ft_seq48', 'moe_rslora_seq48', 'moe_ia3_seq48', 'moe_pos_lora', 'moe_pos_lorma', 'moe_frozen_s48', 'moe_cw_inv', 'gqa_full_ft_seq48', 'gqa_qlora_seq48', 'moe_pos_dom_lora', 'moe_pos_cwinv_lora', 'moe_pos_lora_avg', 'moe_pos_lora_ls', 'moe_pos_lora_seed1', 'moe_pos_lora_seed2',

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded submission_p3s_ensemble_ens.zip to bucket kavincg23/cs781-models
skipping ens_bias (no honest gain)

== per-label F1 of the final out-of-fold ensemble: all / CL / IT (support all / CL / IT) ==
  Fact                     F1 0.748 / 0.746 / 0.753   (4542 / 3228 / 1314)
  Issue                    F1 0.549 / 0.361 / 0.612   (122 / 20 / 102)
  ArgumentPetitioner       F1 0.570 / 0.585 / 0.539   (2013 / 1320 / 693)
  ArgumentRespondent       F1 0.584 / 0.648 / 0.377   (1620 / 1232 / 388)
  Statute                  F1 0.772 / 0.784 / 0.765   (1219 / 425 / 794)
  Dissent                  F1 0.000 / 0.000 / 0.000   (284 / 284 / 0)
  PrecedentReliedUpon      F1 0.532 / 0.523 / 0.552   (2199 / 1447 / 752)
  PrecedentNotReliedUpon   F1 0.279 / 0.192 / 0.313   (619 / 209 / 410)
  PrecedentOverruled       F1 0.000 / 0.000 / 0.000   (3 / 0 / 3)
  RulingByLowerCourt       F1 0.345 / 0.256 / 0.432   (976 / 494 / 482)
  RatioOfTheDecision       F1 0.661 / 0.663 / 0.658   (4704 / 3144 / 1560)
  

MoE random-init frozen 48    191.6        3.68       768      4.1      1.1     29970


... (25 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded summary_p3s_ensemble.json to bucket kavincg23/cs781-models
   /kaggle/working/p3_submissions_p3s_ensemble/p3s_ensemble_ens_prediction.json 39 KB
   /kaggle/working/p3_submissions_p3s_ensemble/submission_p3s_ensemble_ens.zip 1 KB
done. Send back: summary_p3s_ensemble.json and the zips you want to submit.
